# The Card–Krueger benchmark: does raising the minimum wage kill jobs?

On 1 April 1992, New Jersey raised its minimum wage from \$4.25 to \$5.05 an hour. Neighbouring
Pennsylvania kept its minimum at \$4.25. The textbook competitive-market prediction is
unambiguous: a binding wage floor should reduce employment. Card & Krueger (1994) tested it by
surveying 410 fast-food restaurants — Burger King, KFC, Roy Rogers, and Wendy's — on both sides
of the state border, once in February–March 1992 (before the rise) and again in November–December
1992 (after). Because the two states share a labour market and an economic climate, Pennsylvania
serves as the counterfactual for what would have happened to New Jersey employment without the
wage rise.

This is *the* canonical difference-in-differences study, and its result was famously
counterintuitive: employment in New Jersey did **not** fall relative to Pennsylvania — the
published DiD estimate is **+2.76** full-time-equivalent (FTE) jobs per store (Table 3, standard
error 1.36). The benchmark for `formative`'s `DiD` is therefore: feed it the public survey data
and check that it reproduces that coefficient.

## Loading the data

The dataset is distributed on [David Card's data page](https://davidcard.berkeley.edu/data_sets.html)
as a zip containing `public.dat` (one row per store, both survey waves side by side) and a
codebook. Following Card & Krueger, employment is measured in full-time equivalents: full-time
staff plus managers plus half the part-time staff.

In [1]:
import io
import urllib.request
import zipfile

import pandas as pd

URL = "https://davidcard.berkeley.edu/data_sets/njmin.zip"
COLS = [
    "SHEET", "CHAIN", "CO_OWNED", "STATE", "SOUTHJ", "CENTRALJ", "NORTHJ",
    "PA1", "PA2", "SHORE", "NCALLS", "EMPFT", "EMPPT", "NMGRS", "WAGE_ST",
    "INCTIME", "FIRSTINC", "BONUS", "PCTAFF", "MEALS", "OPEN", "HRSOPEN",
    "PSODA", "PFRY", "PENTREE", "NREGS", "NREGS11", "TYPE2", "STATUS2",
    "DATE2", "NCALLS2", "EMPFT2", "EMPPT2", "NMGRS2", "WAGE_ST2", "INCTIME2",
    "FIRSTIN2", "SPECIAL2", "MEALS2", "OPEN2R", "HRSOPEN2", "PSODA2",
    "PFRY2", "PENTREE2", "NREGS2", "NREGS112",
]

with zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(URL).read())) as z:
    raw = pd.read_csv(z.open("public.dat"), sep=r"\s+", header=None,
                      names=COLS, na_values=".")

# FTE employment = full-timers + managers + 0.5 × part-timers (CK's definition)
raw["fte_before"] = raw["EMPFT"] + raw["NMGRS"] + 0.5 * raw["EMPPT"]
raw["fte_after"] = raw["EMPFT2"] + raw["NMGRS2"] + 0.5 * raw["EMPPT2"]

print(f"stores: {len(raw)}")

stores: 410


Before estimating anything, we can check the raw ingredients against the paper. Card & Krueger's
Table 3 reports four cell means (NJ: 20.44 → 21.03; PA: 23.33 → 21.17); the DiD will be built
from exactly these:

In [2]:
for state, name in [(1, "NJ"), (0, "PA")]:
    sub = raw[raw.STATE == state]
    print(f"{name}: before {sub.fte_before.mean():.2f}, after {sub.fte_after.mean():.2f}")

NJ: before 20.44, after 21.03
PA: before 23.33, after 21.17


All four match the published table to the last digit. Note the shape of the problem already
visible here: New Jersey stores were *smaller* than Pennsylvania stores before the wage rise,
so a naive post-period comparison would be biased against New Jersey — the exact baseline
difference DiD exists to remove.

## Running DiD

`DiD` expects long-format data — one row per store per period — with binary group and time
indicators:

In [3]:
from formative.causal import DAG, DiD

long = pd.concat(
    [
        pd.DataFrame({"store": raw.SHEET, "nj": raw.STATE, "after": 0, "fte": raw.fte_before}),
        pd.DataFrame({"store": raw.SHEET, "nj": raw.STATE, "after": 1, "fte": raw.fte_after}),
    ],
    ignore_index=True,
).dropna(subset=["fte"])

dag = DAG()
dag.assume("nj").causes("fte")
dag.assume("after").causes("fte")

result = DiD(dag, group="nj", time="after", outcome="fte").fit(long)
print(result.summary())


DiD Causal Effect: (nj × after) → fte
  Estimand: ATT (average treatment effect on the treated)
──────────────────────────────────────────────────────
  DiD estimate         :     2.7536
  Naive post-diff      :    -0.1382  (treated post − control post)
  Baseline bias removed:    -2.8918

  Std. error           :     1.6884
  95% CI               : [-0.5607, 6.0679]
  p-value              :     0.1033
  N                    :        794

  Assumptions
  ┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄┄
  [ untestable ]  Parallel trends: treated and control groups would have followed the same trend absent treatment
  [ untestable ]  No anticipation: treatment does not affect outcomes before it begins
  [ untestable ]  Stable group composition: group membership does not change due to treatment
  [ untestable ]  Stable Unit Treatment Value Assumption (SUTVA)



The DiD estimate is **+2.75** FTE jobs — Card & Krueger's published figure is 2.76, an agreement
to within a hundredth of an FTE (rounding-level noise; note that the published two-decimal cell
means themselves difference to 2.75). The naive post-period difference of −0.14 also matches the
paper's reported NJ−PA gap of −0.14, and the summary shows DiD stripping out the −2.89 baseline
difference between the states. The sign of the result is the famous finding: no evidence that
the minimum wage rise reduced employment.

Restricting to the balanced panel — the 384 stores with employment measured in both waves —
reproduces the paper's balanced-sample row exactly (published: 2.75, from mean changes of
NJ +0.47 and PA −2.28):

In [4]:
balanced = raw.dropna(subset=["fte_before", "fte_after"]).SHEET
result_bal = DiD(dag, group="nj", time="after", outcome="fte").fit(
    long[long.store.isin(balanced)]
)
print(f"balanced-sample DiD: {result_bal.effect:.4f}")

balanced-sample DiD: 2.7500


## A note on the standard error

`formative` reports an SE of 1.69, computed from the pooled OLS interaction model on
store-periods. Card & Krueger's published 1.36 comes from a different (and for panel data,
sharper) calculation — the variance of the store-level *changes*, which nets out persistent
store-to-store size differences before computing uncertainty. The same data reproduces their
number too (published balanced-sample SE: 1.34):

In [5]:
import numpy as np

change = (raw.fte_after - raw.fte_before).dropna()
nj_change = change[raw.STATE == 1]
pa_change = change[raw.STATE == 0]

did_changes = nj_change.mean() - pa_change.mean()
se_changes = np.sqrt(nj_change.var(ddof=1) / len(nj_change)
                     + pa_change.var(ddof=1) / len(pa_change))
print(f"DiD from store-level changes: {did_changes:.4f}, SE: {se_changes:.4f}")
print(f"mean change — NJ: {nj_change.mean():+.2f}, PA: {pa_change.mean():+.2f}")

DiD from store-level changes: 2.7500, SE: 1.3423
mean change — NJ: +0.47, PA: -2.28


The point estimate is identical either way; the two SEs answer slightly different questions
about it, and `formative`'s is the more conservative of the two.

## Scorecard

In [6]:
scorecard = pd.DataFrame(
    [
        ("Naive post-period NJ−PA difference", -0.14, result.naive_diff),
        ("DiD estimate, all available stores", 2.76, result.effect),
        ("DiD estimate, balanced panel", 2.75, result_bal.effect),
        ("Change-score SE, balanced panel", 1.34, se_changes),
    ],
    columns=["Quantity", "Published", "formative"],
)
scorecard.round(4)

,Quantity,Published,formative
0,Naive post-period NJ−PA difference,-0.14,-0.1382
1,"DiD estimate, all available stores",2.76,2.7536
2,"DiD estimate, balanced panel",2.75,2.7500
3,"Change-score SE, balanced panel",1.34,1.3423


The headline number of the most famous difference-in-differences study in economics — and its
counterintuitive sign — comes straight out of `formative`'s `DiD` on the raw public data.

## References

- Card, D. & Krueger, A. (1994). "Minimum Wages and Employment: A Case Study of the Fast-Food
  Industry in New Jersey and Pennsylvania." *American Economic Review*, 84(4), 772–793.
- Data: [David Card's data page](https://davidcard.berkeley.edu/data_sets.html) (`njmin.zip`).